Built-in function: JVM -> Catalyst Optimiser    
Python UDF: JVM -> serialise row -> python process -> deserialise -> JVM -per row     
Pandas UDF: JVM -> serialise batch -> python process -> deserialise -> JVM -per batch     

- so built-in function is preferred 

built-in function    
**String operations**     
F.trim(), F.lower(), F.upper(), F.regexp_replace(), F.split(), F.concat()

**Math**     
F.round(), F.abs(), F.log(), F.sqrt(), F.pow()

**Date/time**    
F.year(), F.month(), F.dayofweek(), F.date_diff(), F.unix_timestamp()

**Conditional**     
F.when().when().otherwise(), F.coalesce(), F.isnull(), F.isnan()

**Array/map (for complex types)**     
F.explode(), F.collect_list(), F.map_keys(), F.array_contains()

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark =    SparkSession.builder \
    .appName("day13") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.endpoint",           "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key",         "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key",         "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access",  "true") \
    .config("spark.hadoop.fs.s3a.impl",               "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.sql.catalogImplementation",        "hive") \
    .config("hive.metastore.uris",                    "thrift://metastore:9083") \
    .config("spark.eventLog.enabled",                 "true") \
    .config("spark.eventLog.dir",                     "s3a://spark-event/") \
    .config("spark.sql.shuffle.partitions",           str(4)) \
    .config("spark.sql.adaptive.enabled",             "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .enableHiveSupport() \
    .getOrCreate()

In [3]:
spark.sparkContext.setLogLevel("WARN")

transformation

In [16]:
from pyspark.sql import functions as F
from pyspark.sql.functions import udf
from pyspark.sql.functions import pandas_udf
from pyspark.sql.window import Window
from pyspark.sql.types import StringType, DoubleType, IntegerType, LongType
from pyspark import StorageLevel
import pandas as pd
import numpy as np
import time

In [ ]:
# Define — plain Python function
def classify_salary(salary):
    if salary is None or salary <= 0:
        return "Unknown"
    elif salary >= 100000:
        return "Senior"
    elif salary >= 70000:
        return "Mid"
    else:
        return "Junior"

# Register as UDF with return type
classify_udf = udf(classify_salary, StringType())

# Apply — row by row, Python overhead per row
df_with_band = df.withColumn("band", classify_udf("salary"))
df_with_band.groupBy("band").count().show()


# better than above python UDF with built-in function
df.withColumn("band",
    F.when(F.col("salary") >= 100000, "Senior")
     .when(F.col("salary") >= 70000,  "Mid")
     .when(F.col("salary") >  0,      "Junior")
     .otherwise("Unknown")
)


# Classify by salary band — operates on a pd.Series, not a single value
# get input as batch(pd.series) not row like regular python
@pandas_udf(StringType())
def classify_salary_vectorized(salary: pd.Series) -> pd.Series:
    conditions = [
        salary >= 100000,
        salary >= 70000,
        salary > 0,
    ]
    choices = ["Senior", "Mid", "Junior"]
    return pd.Series(np.select(conditions, choices, default="Unknown"))

df_vectorized = employees.withColumn("band", classify_salary_vectorized(F.col("salary")))
df_vectorized.groupBy("band").count().show()

In [6]:
taxi = spark.read.parquet("s3a://spark-warehouse/raw_data/save_from_notebook")
print(taxi.count())

300000


In [7]:
taxi = taxi.filter( (F.col("fare_amount") > 0) & (F.col("total_amount") > 0) )
print(taxi.count())

293059


In [14]:
taxi.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- total_amount_thb: double (nullable = true)
 |-- is_valid_passenger_count: boolean (nullable = true)
 |-- tpep_picku

In [22]:
def classify_payment(payment_type):
    if payment_type == 0:
        return "Flex Fare trip"
    elif payment_type == 1:
        return "Credit card"
    elif payment_type == 2:
        return "Cash"
    elif payment_type == 3:
        return "No charge"
    elif payment_type == 4:
        return "Dispute"
    elif payment_type == 5:
        return "Unknown"
    elif payment_type == 6:
        return "Voided trip"
    else:
        return "No definition"

classify_payment_udf = udf(classify_payment, StringType())   

taxi.withColumn("payment_type_text", classify_payment_udf("payment_type")).show(1)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+---------------------+------------+--------------------+-----------+------------------+----------------+------------------------+----------------+-----------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|total_amount_thb|is_valid_passenger_count|tpep_pickup_date|payment_type_text|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+---------------------+------------+--------------------+-----------+------------------+----------------

In [40]:
# ── IMPORTANT: what Pandas UDFs CANNOT do ────────────────────────────────────
# Pandas UDFs cannot be used with .over() window specs. Only use Pandas UDF for column-level
# This will raise UNSUPPORTED_EXPR_FOR_WINDOW:
#
#   dept_window = Window.partitionBy("dept_id")
#   employees.withColumn("zscore", my_pandas_udf(F.col("salary")).over(dept_window))
#   ↑ FAILS — Pandas UDF receives a pd.Series per partition,
#     but a window function needs to slide across rows within a partition,
#     which is a fundamentally different execution model.
#
# For window-style computation, always use built-in Spark functions:
# (This is the kind of logic that genuinely needs Python — no built-in equivalent)
@pandas_udf(DoubleType())
def scale_fare_vectorized(fare: pd.Series) -> pd.Series:
    # This is pure Pandas math - very fast!
    return fare * 34.5  # Imagine converting USD to THB


taxi_scored = taxi \
    .withColumn("fare_thb", F.round(scale_fare_vectorized(F.col("total_amount")), 2) )

print("── Salary outliers by department ──────────────────────")
taxi_scored.select("VendorID", "PULocationID", "tpep_pickup_datetime", "total_amount", "fare_thb") \
    .orderBy("fare_thb", ) \
    .show()

── Salary outliers by department ──────────────────────
+--------+------------+--------------------+------------+--------+
|VendorID|PULocationID|tpep_pickup_datetime|total_amount|fare_thb|
+--------+------------+--------------------+------------+--------+
|       1|          80| 2025-05-03 03:43:40|        1.01|   34.85|
|       1|           1| 2025-05-01 14:50:17|        1.01|   34.85|
|       1|         261| 2025-05-01 17:56:55|        1.01|   34.85|
|       2|         136| 2025-05-01 20:51:50|        1.01|   34.85|
|       2|         132| 2025-05-01 03:27:36|        1.05|   36.23|
|       2|          25| 2025-05-03 01:21:13|         1.2|    41.4|
|       1|         170| 2025-05-01 18:18:24|        1.76|   60.72|
|       1|         230| 2025-05-03 13:54:25|        1.76|   60.72|
|       1|          88| 2025-05-02 17:14:07|        1.76|   60.72|
|       1|         230| 2025-05-03 13:55:52|        1.76|   60.72|
|       1|         148| 2025-05-03 14:43:57|        1.76|   60.72|
|     

In [25]:
# Without broadcast: dict is serialised into every task (hundreds of copies)
# With broadcast: dict is sent once to each executor (N copies where N = worker count)

rateCode_lookup = {
    1: "Standard rate",
    2: "JFK",
    3: "Newark",
    4: "Nassau or Westchester", 
    5: "Negotiated fare",
    6: "Group ride",
    99: "Null/unknown"
}

# Broadcast to all executors
broadcast_lookup = spark.sparkContext.broadcast(rateCode_lookup)

@udf(StringType())
def expand_rateCode(code):
    # .value accesses the broadcast object on the executor
    return broadcast_lookup.value.get(code, "No definition")

taxi.withColumn("dept_full", expand_rateCode(F.col("RatecodeID"))) \
         .select("tpep_pickup_datetime", "RatecodeID", "dept_full") \
         .orderBy("tpep_pickup_datetime") \
         .show(10)

# Clean up when done — frees executor memory
broadcast_lookup.unpersist()

+--------------------+----------+-------------+
|tpep_pickup_datetime|RatecodeID|    dept_full|
+--------------------+----------+-------------+
| 2025-04-30 22:39:14|         1|Standard rate|
| 2025-04-30 22:53:51|         1|Standard rate|
| 2025-04-30 23:04:04|         1|Standard rate|
| 2025-04-30 23:40:28|         1|Standard rate|
| 2025-04-30 23:47:53|         1|Standard rate|
| 2025-04-30 23:50:34|         1|Standard rate|
| 2025-04-30 23:52:58|         1|Standard rate|
| 2025-04-30 23:53:23|         1|Standard rate|
| 2025-04-30 23:54:36|         1|Standard rate|
| 2025-04-30 23:55:34|         1|Standard rate|
+--------------------+----------+-------------+
only showing top 10 rows



In [31]:
taxi_zone = spark.read \
    .option("header", "true").option("inferSchema", "true") \
    .csv("s3a://metadata-rawdata/taxi_zone_lookup.csv")

# This join + filter is expensive — reads from Hive, joins, filters
start = time.time()
cleaned = taxi \
    .filter(F.col("is_valid_passenger_count") == True) \
    .join(F.broadcast(taxi_zone), taxi["PULocationID"] == taxi_zone["LocationID"], how="left") \
    .withColumn("Vendor", \
         F.when(F.col("VendorID") == 1, "Creative Mobile Technologies LLC") \
         .when(F.col("VendorID") == 2,  "Curb Mobility LLC") \
         .when(F.col("VendorID") == 6,  "Myle Technologies Inc") \
         .when(F.col("VendorID") == 7,  "Helix") \
         .otherwise("No definition") \
    )
end = time.time()
print(f"read data: {end - start:.4f} seconds")

# Without cache: the join + filter runs TWICE
start = time.time()
cleaned.groupBy("VendorID").count()         # run 1
cleaned.groupBy("tpep_pickup_date").agg(F.avg("fare_amount")).show(5) # run 2
end = time.time()
print(f"without cache: {end - start:.4f} seconds")

# ── With cache ────────────────────────────────────────────────────────────────
cleaned.cache()         # declares intent — data not yet materialised

# First action triggers materialisation — data written to executor memory
cleaned.count()         # now cached
print("Cached — subsequent actions read from memory, not S3 + Hive")

start = time.time()
cleaned.groupBy("VendorID").count()          # reads from cache
cleaned.groupBy("tpep_pickup_date").agg(F.avg("fare_amount")).show(5)  # reads from cache
end = time.time()
print(f"with cache: {end - start:.4f} seconds")

# Always unpersist when done — executor memory is shared with running tasks
cleaned.unpersist()
print("Cache freed")

read data: 0.0293 seconds
+----------------+------------------+
|tpep_pickup_date|  avg(fare_amount)|
+----------------+------------------+
|      2025-05-02|19.370404272831266|
|      2025-05-01|20.257219273708916|
|      2025-04-30|14.934999999999997|
|      2025-05-03|17.534186050666463|
+----------------+------------------+

without cache: 1.2184 seconds
Cached — subsequent actions read from memory, not S3 + Hive
+----------------+------------------+
|tpep_pickup_date|  avg(fare_amount)|
+----------------+------------------+
|      2025-05-02|19.370404272831266|
|      2025-05-01|20.257219273708916|
|      2025-04-30|14.934999999999997|
|      2025-05-03|17.534186050666463|
+----------------+------------------+

with cache: 0.3313 seconds
Cache freed


In [ ]:
# MEMORY_ONLY (default for .cache())
# Stores as deserialized JVM objects. Fastest to read. Uses most memory.
# If it doesn't fit: some partitions are recomputed on access (not evicted to disk)
cleaned.persist(StorageLevel.MEMORY_ONLY)

# MEMORY_AND_DISK
# Spills partitions that don't fit in memory to local disk.
# Slower than memory-only, but no recomputation if memory is tight.
cleaned.persist(StorageLevel.MEMORY_AND_DISK)

# DISK_ONLY
# Entire cache on local executor disk. Useful for very large DataFrames.
# Slower than memory, but always available without recomputation.
cleaned.persist(StorageLevel.DISK_ONLY)

# MEMORY_AND_DISK_2 / MEMORY_ONLY_2
# Replicates cached data to 2 executors. Fault tolerant if a worker dies.
# Useful when you can't afford to recompute on failure.
cleaned.persist(StorageLevel.MEMORY_AND_DISK_2)

In [ ]:
# Bad — used only once, caching costs more than it saves
cleaned.cache()
result = cleaned.count()
cleaned.unpersist()

# Bad — the DataFrame is cheap to recompute (just a read, no joins)
raw = spark.read.parquet("s3a://raw-data/yellow_tripdata_2025*/")
raw.cache()              # pointless — S3 reads are cached at the OS level anyway

# Good — expensive computation used multiple times
enriched = raw \
    .join(taxi_zone, raw["PULocationID"] == taxi_zone["LocationID"], how="left") \  # expensive
    .withColumn("Vendor", \
         F.when(F.col("VendorID") == 1, "Creative Mobile Technologies LLC") \
         .when(F.col("VendorID") == 2,  "Curb Mobility LLC") \
         .when(F.col("VendorID") == 6,  "Myle Technologies Inc") \
         .when(F.col("VendorID") == 7,  "Helix") \
         .otherwise("No definition") ) \            
    .withColumn("Ratecode_full", expand_rateCode(F.col("RatecodeID")))  # multiple transforms
enriched.cache()
enriched.count()                 # materialise
r1 = enriched.groupBy("VendorID").count()
r2 = enriched.join(taxi_zone, raw["DOLocationID"] == taxi_zone["LocationID"], how="left")
enriched.unpersist()

In [33]:
# ── Explain modes ────────────────────────────────────────────────────────────
print("-----------------------Simple — physical plan only (most useful day-to-day)-----------------------")
cleaned.groupBy("Vendor").count().explain()

print("-----------------Extended — parsed, analysed, optimised logical + physical plans-------------------")
cleaned.groupBy("Vendor").count().explain(extended=True)

print("-----------------------Cost — physical plan with cost estimates (AQE info)-----------------------")
cleaned.groupBy("Vendor").count().explain(mode="cost")

print("-----------------Formatted — cleaner indented tree structure (Spark 3.x+)-----------------------")
cleaned.groupBy("Vendor").count().explain(mode="formatted")

-----------------------Simple — physical plan only (most useful day-to-day)-----------------------
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[Vendor#7034], functions=[count(1)])
   +- Exchange hashpartitioning(Vendor#7034, 4), ENSURE_REQUIREMENTS, [plan_id=1595]
      +- HashAggregate(keys=[Vendor#7034], functions=[partial_count(1)])
         +- Project [CASE WHEN (VendorID#0 = 1) THEN Creative Mobile Technologies LLC WHEN (VendorID#0 = 2) THEN Curb Mobility LLC WHEN (VendorID#0 = 6) THEN Myle Technologies Inc WHEN (VendorID#0 = 7) THEN Helix ELSE No definition END AS Vendor#7034]
            +- BroadcastHashJoin [PULocationID#7], [LocationID#6973], LeftOuter, BuildRight, false
               :- Project [VendorID#0, PULocationID#7]
               :  +- Filter (((((isnotnull(fare_amount#10) AND isnotnull(total_amount#15)) AND isnotnull(is_valid_passenger_count#20)) AND (fare_amount#10 > 0.0)) AND (total_amount#15 > 0.0)) AND is_valid_passenger_count#2

FileScan parquet          → reading Parquet files from S3/disk
                            check PartitionFilters: — are partitions being pruned?
                            check PushedFilters:    — is the filter pushed to the file scan?

Filter                    → applying a WHERE condition
                            if this appears ABOVE FileScan, the filter wasn't pushed down
                            if it appears INSIDE FileScan as PushedFilters, it was (better)

Project                   → SELECT — choosing which columns to keep

Exchange                  → SHUFFLE — data moved across the network between executors
                            this is expensive, minimise it
                            appears before groupBy, join (SortMergeJoin), repartition

HashAggregate             → groupBy using a hash table (efficient)
                            appears TWICE for groupBy: once per executor (partial),
                            once after the shuffle (final)
                            the double appearance is a Spark optimisation — pre-aggregating
                            on each executor before sending across the network

SortMergeJoin             → join between two large tables
                            both tables are sorted then merged — requires two Exchange nodes
                            use broadcast join when one table is small to avoid this

BroadcastHashJoin         → join where small table was broadcast to all executors
                            no Exchange, no sort — just a hash lookup
                            ideal for fact × dimension joins

BroadcastExchange         → the broadcast itself — small table being sent to executors
                            should appear alongside BroadcastHashJoin

Sort                      → ORDER BY or pre-sort for SortMergeJoin
                            always comes with an Exchange before it (sort needs data local)

In [37]:
print("-------------------------------------1----------------------------------------------")
# 1. Confirm broadcast join
taxi.join(taxi_zone, taxi["PULocationID"] == taxi_zone["LocationID"], how="left").explain()
# Should see: BroadcastHashJoin, BroadcastExchange
# Should NOT see: SortMergeJoin, Exchange (shuffle)
print("-------------------------------------2----------------------------------------------")
# 2. Force SortMergeJoin and compare
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
taxi.join(taxi_zone, taxi["PULocationID"] == taxi_zone["LocationID"], how="left").explain()
# Should see: SortMergeJoin + two Exchange nodes (one per table)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10485760)
print("-------------------------------------3----------------------------------------------")
# 3. Check partition pruning
partitioned_taxi = spark.read.parquet("s3a://spark-warehouse/raw_data/multiple_col_partition/")
partitioned_taxi.filter((F.col("year") == 2025) & (F.col("month") == 9)).explain()
# Look for: PartitionFilters: [isnotnull(year), (year = 2024), (month = 10)]
print("-------------------------------------4----------------------------------------------")
# 4. Count exchanges in a complex pipeline
taxi.join(taxi_zone, taxi["PULocationID"] == taxi_zone["LocationID"], how="left") \
    .groupBy("VendorID").agg(F.avg("fare_amount").alias("avg_fare")) \
    .orderBy("avg_fare").explain()                                           
# Count the 'Exchange' nodes — each one is a network shuffle
print("-------------------------------------5----------------------------------------------")
# 5. AQE (Adaptive Query Execution) — Spark 3.x optimising at runtime
# When spark.sql.adaptive.enabled = true (set in spark_session.py),
# Spark can change the plan AFTER seeing actual data sizes.
# Example: if a join table turns out to be small, Spark switches to broadcast
# even if it wasn't originally planned that way.
# You'll see "AdaptiveSparkPlan" wrapping the plan when AQE is active.
spark.sql("SELECT * FROM local_db.sample_hive_table").explain(mode="formatted")

-------------------------------------1----------------------------------------------
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [PULocationID#7], [LocationID#6973], LeftOuter, BuildRight, false
   :- Filter (((isnotnull(fare_amount#10) AND isnotnull(total_amount#15)) AND (fare_amount#10 > 0.0)) AND (total_amount#15 > 0.0))
   :  +- FileScan parquet [VendorID#0,tpep_pickup_datetime#1,tpep_dropoff_datetime#2,passenger_count#3L,trip_distance#4,RatecodeID#5L,store_and_fwd_flag#6,PULocationID#7,DOLocationID#8,payment_type#9L,fare_amount#10,extra#11,mta_tax#12,tip_amount#13,improvement_surcharge#14,total_amount#15,congestion_surcharge#16,Airport_fee#17,cbd_congestion_fee#18,total_amount_thb#19,is_valid_passenger_count#20,tpep_pickup_date#21] Batched: true, DataFilters: [isnotnull(fare_amount#10), isnotnull(total_amount#15), (fare_amount#10 > 0.0), (total_amount#15 ..., Format: Parquet, Location: InMemoryFileIndex(1 paths)[s3a://spark-warehouse/raw_data/save_

**exercise1**

In [41]:
taxi = spark.read.parquet("s3a://raw-data/yellow_tripdata_2025-10.parquet")
taxi.cache()
taxi.count()  # materialise cache

# Regular UDF — row by row, Python overhead per row
@udf(StringType())
def fare_band_udf(fare):
    if fare is None: return "unknown"
    return "high" if fare > 30 else "mid" if fare > 15 else "low"

# Pandas UDF — batch processing, faster than regular UDF
# Note: this specific logic (simple thresholds) could also be F.when(),
# but pd.cut with arbitrary uneven bins is a case where Pandas UDF
# is more expressive than chaining many F.when() conditions
@pandas_udf(StringType())
def fare_band_pandas(fare: pd.Series) -> pd.Series:
    bins   = [-np.inf, 10, 20, 35, 60, np.inf]
    labels = ["very_low", "low", "mid", "high", "premium"]
    return pd.cut(fare, bins=bins, labels=labels).astype(str)

# Built-in — always try this first, fastest of the three
fare_band_builtin = (
    F.when(F.col("fare_amount") > 30, "high")
     .when(F.col("fare_amount") > 15, "mid")
     .otherwise("low")
)

for name, expr in [
    ("Regular UDF",  fare_band_udf(F.col("fare_amount"))),
    ("Pandas UDF",   fare_band_pandas(F.col("fare_amount"))),
    ("Built-in",     fare_band_builtin),
]:
    t = time.time()
    taxi.withColumn("band", expr).groupBy("band").count().show()
    print(f"{name}: {time.time() - t:.2f}s\n")

taxi.unpersist()

+----+-------+
|band|  count|
+----+-------+
|high| 715614|
| mid|1250229|
| low|2462856|
+----+-------+

Regular UDF: 6.30s

+--------+-------+
|    band|  count|
+--------+-------+
|    high| 336766|
|     mid| 788757|
|     low|1573858|
| premium| 218080|
|very_low|1511238|
+--------+-------+

Pandas UDF: 1.94s

+----+-------+
|band|  count|
+----+-------+
|high| 715614|
| mid|1250229|
| low|2462856|
+----+-------+

Built-in: 0.60s



DataFrame[VendorID: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: bigint, trip_distance: double, RatecodeID: bigint, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double, cbd_congestion_fee: double]

**exercise2**

In [42]:
taxi = spark.read.parquet("s3a://raw-data/yellow_tripdata_2025-10.parquet") \
           .filter(F.col("fare_amount") > 0) \
           .withColumn("hour", F.hour("tpep_pickup_datetime")) \
           .withColumn("trip_mins",
               (F.unix_timestamp("tpep_dropoff_datetime") -
                F.unix_timestamp("tpep_pickup_datetime")) / 60)

# Without cache — Spark re-reads from S3 and recomputes for each action
t = time.time()
r1 = taxi.groupBy("hour").count().collect()
r2 = taxi.groupBy("hour").agg(F.avg("fare_amount")).collect()
r3 = taxi.filter(F.col("trip_mins") > 60).count()
print(f"Without cache: {time.time() - t:.2f}s")

# With cache
taxi.cache()
taxi.count()  # materialise

t = time.time()
r1 = taxi.groupBy("hour").count().collect()
r2 = taxi.groupBy("hour").agg(F.avg("fare_amount")).collect()
r3 = taxi.filter(F.col("trip_mins") > 60).count()
print(f"With cache: {time.time() - t:.2f}s")

taxi.unpersist()

Without cache: 2.08s
With cache: 1.05s


DataFrame[VendorID: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: bigint, trip_distance: double, RatecodeID: bigint, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double, cbd_congestion_fee: double, hour: int, trip_mins: double]

**exercise3**     
broadcast use 1 exchange    
sortmergejoin use 2 exchange

In [43]:
result = taxi \
    .withColumn("hour", F.hour("tpep_pickup_datetime")) \
    .groupBy("hour") \
    .agg(F.avg("fare_amount").alias("avg_fare")) \
    .orderBy("avg_fare", ascending=False)

result.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (9)
+- Sort (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Exchange (5)
            +- HashAggregate (4)
               +- Project (3)
                  +- Filter (2)
                     +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [tpep_pickup_datetime#11916, fare_amount#11925]
Batched: true
Location: InMemoryFileIndex [s3a://raw-data/yellow_tripdata_2025-10.parquet]
PushedFilters: [IsNotNull(fare_amount), GreaterThan(fare_amount,0.0)]
ReadSchema: struct<tpep_pickup_datetime:timestamp_ntz,fare_amount:double>

(2) Filter
Input [2]: [tpep_pickup_datetime#11916, fare_amount#11925]
Condition : (isnotnull(fare_amount#11925) AND (fare_amount#11925 > 0.0))

(3) Project
Output [2]: [fare_amount#11925, hour(tpep_pickup_datetime#11916, Some(Etc/UTC)) AS hour#14518]
Input [2]: [tpep_pickup_datetime#11916, fare_amount#11925]

(4) HashAggregate
Input [2]: [fare_amount#11925, hour#14518]
Keys [1]: [hour#14518]
Functions [1]: [partial_avg

In [44]:
zone_lookup = spark.createDataFrame([
    (1, "EWR"), (132, "JFK"), (138, "LaGuardia")
], ["LocationID", "Zone"])

result2 = taxi \
    .groupBy("PULocationID") \
    .agg(F.count("*").alias("trips")) \
    .join(F.broadcast(zone_lookup),
          taxi["PULocationID"] == zone_lookup["LocationID"],
          how="left")

result2.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (11)
+- BroadcastHashJoin LeftOuter BuildRight (10)
   :- HashAggregate (6)
   :  +- Exchange (5)
   :     +- HashAggregate (4)
   :        +- Project (3)
   :           +- Filter (2)
   :              +- Scan parquet  (1)
   +- BroadcastExchange (9)
      +- Filter (8)
         +- Scan ExistingRDD (7)


(1) Scan parquet 
Output [2]: [PULocationID#11922, fare_amount#11925]
Batched: true
Location: InMemoryFileIndex [s3a://raw-data/yellow_tripdata_2025-10.parquet]
PushedFilters: [IsNotNull(fare_amount), GreaterThan(fare_amount,0.0)]
ReadSchema: struct<PULocationID:int,fare_amount:double>

(2) Filter
Input [2]: [PULocationID#11922, fare_amount#11925]
Condition : (isnotnull(fare_amount#11925) AND (fare_amount#11925 > 0.0))

(3) Project
Output [1]: [PULocationID#11922]
Input [2]: [PULocationID#11922, fare_amount#11925]

(4) HashAggregate
Input [1]: [PULocationID#11922]
Keys [1]: [PULocationID#11922]
Functions [1]: [partial_count(1)]
Aggregate Attributes

**exercise4**

In [45]:
taxi_part = spark.read.parquet("s3a://spark-warehouse/raw_data/multiple_col_partition/")

print("---------------------------------------------partition index---------------------------------------------")
# Partition filter — pushed to file selection (skips entire folders)
taxi_part.filter(F.col("month") == 10).explain(mode="formatted")
# Look for: PartitionFilters in FileScan
print("------------------------------------------non-partition index--------------------------------------------")
# Column filter — pushed into Parquet page/row-group scanning
taxi_part.filter(F.col("fare_amount") > 50).explain(mode="formatted")
# Look for: PushedFilters in FileScan
print("------------------------------------------------both-----------------------------------------------------")
# Both together
taxi_part.filter(
    (F.col("month") == 10) & (F.col("fare_amount") > 50)
).explain(mode="formatted")

---------------------------------------------partition index---------------------------------------------
== Physical Plan ==
* ColumnarToRow (2)
+- Scan parquet  (1)


(1) Scan parquet 
Output [22]: [VendorID#14611, tpep_pickup_datetime#14612, tpep_dropoff_datetime#14613, passenger_count#14614L, trip_distance#14615, RatecodeID#14616L, store_and_fwd_flag#14617, PULocationID#14618, DOLocationID#14619, payment_type#14620L, fare_amount#14621, extra#14622, mta_tax#14623, tip_amount#14624, tolls_amount#14625, improvement_surcharge#14626, total_amount#14627, congestion_surcharge#14628, Airport_fee#14629, cbd_congestion_fee#14630, year#14631, month#14632]
Batched: true
Location: InMemoryFileIndex [s3a://spark-warehouse/raw_data/multiple_col_partition]
PartitionFilters: [isnotnull(month#14632), (month#14632 = 10)]
ReadSchema: struct<VendorID:int,tpep_pickup_datetime:timestamp_ntz,tpep_dropoff_datetime:timestamp_ntz,passenger_count:bigint,trip_distance:double,RatecodeID:bigint,store_and_fwd_fla

In [46]:
spark.stop()